<a href="https://colab.research.google.com/github/Pakin49/current/blob/main/pattern-recognition/python/KERAS2_with_GridSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install scikeras tensorflow keras scikit-learn==1.4.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 50.5 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.


In [2]:
import keras
from keras.models import Sequential
from keras.layers import Dense
import numpy as np

from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV
import pandas as pd

In [3]:
def create_network(optimizer="rmsprop"):
    #Define the model achitecture
    model = Sequential()
    model.add(Dense(512, input_shape=(784,),activation='relu'))
    #model.add(Dropout(0.2))
    model.add(Dense(512,activation='relu'))
    #model.add(Dropout(0.2))
    #model.add(Dropout(0.2))
    model.add(Dense(10,activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=["accuracy"])
    model.summary()
    return model
#Make the model learn
#MORE OPTIMIZER DETAILS here https://keras.io/optimizers/

In [4]:
from google.colab import files
uploaded =files.upload()
import io
data=pd.read_csv(io.BytesIO(uploaded['digit.csv']),header=None)


Saving digit.csv to digit.csv


In [5]:

X=data.values
X=X/256
from sklearn.model_selection import train_test_split
X_train , X_test, Y_train, Y_test=train_test_split(X[:,:784],X[:,784],test_size=0.2)
y_train = keras.utils.to_categorical(Y_train, num_classes=10)
y_test = keras.utils.to_categorical(Y_test, num_classes=10)

In [6]:
network=KerasClassifier(build_fn=create_network,verbose=1)
epochs=[5,10]
batches=[10,50,100]
optimizer=["rmsprop","adam"]


param_grid = dict(epochs=epochs,batch_size=batches,optimizer=optimizer)
grid = GridSearchCV(estimator=network, param_grid=param_grid, n_jobs=-1,)
grid_result = grid.fit(X_train,y_train)


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │       401,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 669,706 (2.55 MB)

 Trainable params: 669,706 (2.55 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8950 - loss: 0.2546
Epoch 2/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 1.0000 - loss: 1.1453e-04
Epoch 3/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 4.9432e-05
Epoch 4/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 1.7075e-05
Epoch 5/5
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 1.0000 - loss: 1.1985e-05


In [8]:
# (784+1) * 512 = 401,920
# (512+1) * 512 = 262,656
# (512+1) * 10  = 5,130

In [7]:
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

Best: 1.000000 using {'batch_size': 10, 'epochs': 5, 'optimizer': 'rmsprop'}
1.000000 (0.000000) with: {'batch_size': 10, 'epochs': 5, 'optimizer': 'rmsprop'}
1.000000 (0.000000) with: {'batch_size': 10, 'epochs': 5, 'optimizer': 'adam'}
1.000000 (0.000000) with: {'batch_size': 10, 'epochs': 10, 'optimizer': 'rmsprop'}
1.000000 (0.000000) with: {'batch_size': 10, 'epochs': 10, 'optimizer': 'adam'}
1.000000 (0.000000) with: {'batch_size': 50, 'epochs': 5, 'optimizer': 'rmsprop'}
1.000000 (0.000000) with: {'batch_size': 50, 'epochs': 5, 'optimizer': 'adam'}
1.000000 (0.000000) with: {'batch_size': 50, 'epochs': 10, 'optimizer': 'rmsprop'}
1.000000 (0.000000) with: {'batch_size': 50, 'epochs': 10, 'optimizer': 'adam'}
1.000000 (0.000000) with: {'batch_size': 100, 'epochs': 5, 'optimizer': 'rmsprop'}
1.000000 (0.000000) with: {'batch_size': 100, 'epochs': 5, 'optimizer': 'adam'}
1.000000 (0.000000) with: {'batch_size': 100, 'epochs': 10, 'optimizer': 'rmsprop'}
1.000000 (0.000000) with: {'